# Handling with the five EHR-related datasets with categorical columns

## Import all the necessary libraries

In [1]:
import numpy as np

In [2]:
from pandas import read_csv, DataFrame

In [3]:
import pandas as pd

In [4]:
from sklearn.preprocessing import MinMaxScaler

In [115]:
from typing import Tuple, List, Set, Dict, Any

In [6]:
from regex import match

In [7]:
import os

In [10]:
# explicitly require this experimental feature
from sklearn.experimental import enable_iterative_imputer  # noqa
# now you can import normally from sklearn.impute
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.ensemble import RandomForestRegressor
import torch
from filelock import FileLock

Please use the T4 GPU on google colab

In [12]:
from multiprocessing import Pool, cpu_count

n_cpus = cpu_count()
print(f"Number of CPUs: {n_cpus}")

Number of CPUs: 10


## Write the reusable utility functions

In [13]:
def identify_binary_and_numerical_features(df: DataFrame) -> Tuple[List[str], List[str]]:
     # 1. Identify “binary” columns (unique values ⊆ {0,1})
    categorical_cols = [col for col in df.columns if match(r".+-is_.+", col)]

    # 2. All the other numeric columns
    numeric_cols = list(set(df.columns) - set(categorical_cols))

    # return to a tuple of two lists of column names
    return numeric_cols, categorical_cols

In [14]:
def normalize_numerical_features(df: DataFrame, num_features: List[str]) -> DataFrame:
    # Create a MinMaxScaler instance
    scaler = MinMaxScaler()

    # Fit and transform the numerical features to the range [0, 1]
    df[list(num_features)] = scaler.fit_transform(df[num_features])

    return df

### Make a test of the two utility functions above

In [28]:
appedicities_MAR_15per_missing_dataset = read_csv("./amputed_datasets/appendicitis_data/MAR_15_perc_5.csv")

In [29]:
appedicities_MAR_15per_missing_dataset = read_csv("./amputed_datasets/appendicitis_data/MNAR_15_perc_5.csv")

In [30]:
feat_m1, feat_c1 = identify_binary_and_numerical_features(appedicities_MAR_15per_missing_dataset)

In [31]:
feat_m2, feat_c2 = identify_binary_and_numerical_features(appedicities_MAR_15per_missing_dataset)

In [32]:
feat_m1

['Neutrophil_Percentage',
 'Height',
 'Paedriatic_Appendicitis_Score',
 'RBC_Count',
 'Appendix_Diameter',
 'BMI',
 'WBC_Count',
 'Hemoglobin',
 'Length_of_Stay',
 'CRP',
 'Weight',
 'Thrombocyte_Count',
 'Age',
 'Body_Temperature',
 'RDW',
 'Alvarado_Score']

In [33]:
feat_m1==feat_m2

True

In [34]:
feat_c1

['Sex-is_female',
 'Sex-is_male',
 'Appendix_on_US-is_yes',
 'Appendix_on_US-is_no',
 'Migratory_Pain-is_no',
 'Migratory_Pain-is_yes',
 'Lower_Right_Abd_Pain-is_yes',
 'Lower_Right_Abd_Pain-is_no',
 'Contralateral_Rebound_Tenderness-is_yes',
 'Contralateral_Rebound_Tenderness-is_no',
 'Coughing_Pain-is_no',
 'Coughing_Pain-is_yes',
 'Nausea-is_no',
 'Nausea-is_yes',
 'Loss_of_Appetite-is_yes',
 'Loss_of_Appetite-is_no',
 'Neutrophilia-is_no',
 'Neutrophilia-is_yes',
 'Ketones_in_Urine-is_++',
 'Ketones_in_Urine-is_no',
 'Ketones_in_Urine-is_+++',
 'Ketones_in_Urine-is_+',
 'RBC_in_Urine-is_+',
 'RBC_in_Urine-is_no',
 'RBC_in_Urine-is_++',
 'RBC_in_Urine-is_+++',
 'WBC_in_Urine-is_no',
 'WBC_in_Urine-is_+',
 'WBC_in_Urine-is_+++',
 'WBC_in_Urine-is_++',
 'Dysuria-is_no',
 'Dysuria-is_yes',
 'Stool-is_normal',
 'Stool-is_constipation',
 'Stool-is_diarrhea',
 'Stool-is_constipation, diarrhea',
 'Peritonitis-is_no',
 'Peritonitis-is_local',
 'Peritonitis-is_generalized',
 'Psoas_Sign-is_y

In [35]:
feat_c1 == feat_c2

True

In [36]:
%%bash
ls preprocessed_datasets/*

preprocessed_datasets/HCV_Egyptian_data_complete.csv
preprocessed_datasets/HCV_data_imputed.csv
preprocessed_datasets/HCV_data_unimputed.csv
preprocessed_datasets/Heart_Failure_Clinical_Records_complete.csv
preprocessed_datasets/Indian_liver_patients_imputed.csv
preprocessed_datasets/Indian_liver_patients_unimputed.csv
preprocessed_datasets/Regensburg_Pediatric_Appendicitis_imputed.csv
preprocessed_datasets/Regensburg_Pediatric_Appendicitis_unimputed.csv
preprocessed_datasets/Vehicle_imputed.csv
preprocessed_datasets/Vehicle_unimputed.csv


In [37]:
normalized_appendicities_df = normalize_numerical_features(appedicities_MAR_15per_missing_dataset, feat_m1)

In [38]:
normalized_appendicities_df[list(feat_m1)].describe()

,Neutrophil_Percentage,Height,Paedriatic_Appendicitis_Score,RBC_Count,Appendix_Diameter,BMI,WBC_Count,Hemoglobin,Length_of_Stay,CRP,Weight,Thrombocyte_Count,Age,Body_Temperature,RDW,Alvarado_Score
count,609.000000,611.000000,596.000000,599.000000,592.000000,616.000000,615.000000,594.000000,601.000000,606.000000,618.000000,597.000000,622.000000,610.000000,611.000000,613.000000
mean,0.633445,0.680853,0.498674,0.401710,0.410265,0.448358,0.374131,0.596553,0.211314,0.112372,0.406082,0.401745,0.605929,0.790126,0.310420,0.568810
std,0.212442,0.142971,0.184512,0.123188,0.172068,0.154212,0.192002,0.125500,0.125911,0.136589,0.169810,0.140130,0.187620,0.064563,0.147212,0.213036
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.455224,0.591241,0.400000,0.315603,0.292035,0.333515,0.220884,0.514881,0.153846,0.033662,0.275044,0.308204,0.494826,0.755725,0.207547,0.400000
50%,0.667164,0.686131,0.500000,0.400709,0.397387,0.426638,0.341365,0.595238,0.153846,0.056240,0.384886,0.390244,0.613562,0.786260,0.283019,0.600000
75%,0.804478,0.788434,0.600000,0.475177,0.509459,0.542548,0.510040,0.678571,0.230769,0.128489,0.527680,0.485588,0.746187,0.824427,0.377358,0.700000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


the test is successful

<hr/>

Let's remind ourselves that, in the next part of this study (VAEQL implementation, we will discretize the <code>[0, 1]</code> normalized the numerical features into 101 states to faciliate the Q-learning) using the <code>round</code> method

<hr/>

### Create the utility functions that compares the processed unimputed/imputed datasets and the amputed dataset to create the missingness masks

In [39]:
def generate_masks_for_missingness(
    original_df: pd.DataFrame,
    amputed_df: pd.DataFrame,
    num_feats: Set[str],
    cat_feats: Set[str],
    imputed_original_df: pd.DataFrame = None,
) -> Tuple[np.ndarray, np.ndarray]:

    # Check if the dataframes match in shape
    if not original_df.shape == imputed_original_df.shape == amputed_df.shape:
        raise Exception("Sorry, the three dataframes do not match in shape")

    # Check if the dataframes have identical column names
    if not set(original_df.columns) == set(imputed_original_df.columns) == set(amputed_df.columns):
        raise Exception("Sorry, the three dataframes do not match in column names")

    num_feats = list(num_feats)
    cat_feats = list(cat_feats)

    # Initialize the maps for numerical and categorical features
    num_feat_map = np.zeros(original_df[num_feats].shape, dtype=int)
    cat_feat_map = np.zeros(original_df[cat_feats].shape, dtype=int)

    # Generate the missingness map for numerical features
    for i, feat in enumerate(num_feats):
        original_col = original_df[feat]
        amputed_col = amputed_df[feat]

        # 1: missing in the original dataset, regardless of whether it is amputed
        num_feat_map[:, i] = np.where(original_col.isna(), 1, 0)  # 1 if missing originally, otherwise 0
        # 2: existing in the original dataset and amputed
        num_feat_map[:, i] = np.where(amputed_col.isna(), 2, num_feat_map[:, i])  # 2 if amputed

    # Generate the missingness map for one-hot-encoded categorical features
    for i, feat in enumerate(cat_feats):
        original_col = original_df[feat]
        amputed_col = amputed_df[feat]

        # 1: missing in the original dataset, regardless of whether it is amputed
        cat_feat_map[:, i] = np.where(original_col.isna(), 1, 0)  # 1 if missing originally, otherwise 0
        # 2: existing in the original dataset and amputed
        cat_feat_map[:, i] = np.where(amputed_col.isna(), 2, cat_feat_map[:, i])  # 2 if amputed

    # Return the maps as a tuple of two arrays
    return num_feat_map, cat_feat_map

### Make a test of the two utility functions above

#### Testing <code>generate_masks_for_missingnes</code> using the <code>Regensburg Appenticitis Dataset</code> (with pre-existing missingness)

##### Read the datasets

In [40]:
%%bash
ls *datasets

amputed_datasets:
HCV_Egyptian_patients
HCV_data
Indian_liver_patients
appendicitis_data
heart_failure_clinical_records

preprocessed_datasets:
HCV_Egyptian_data_complete.csv
HCV_data_imputed.csv
HCV_data_unimputed.csv
Heart_Failure_Clinical_Records_complete.csv
Indian_liver_patients_imputed.csv
Indian_liver_patients_unimputed.csv
Regensburg_Pediatric_Appendicitis_imputed.csv
Regensburg_Pediatric_Appendicitis_unimputed.csv
Vehicle_imputed.csv
Vehicle_unimputed.csv

preprocessing_Awan_2022_datasets:
datasets_masking_and_normalization.ipynb
individual_datasets_preprocessing.ipynb


In [42]:
%%bash
ls amputed_datasets/appendicitis_data | head -n 5

MAR_10_perc_1.csv
MAR_10_perc_10.csv
MAR_10_perc_2.csv
MAR_10_perc_3.csv
MAR_10_perc_4.csv


In [44]:
test_apd_df_MAR = pd.read_csv("amputed_datasets/appendicitis_data/MAR_15_perc_8.csv")
test_apd_df_MNAR = pd.read_csv("amputed_datasets/appendicitis_data/MNAR_20_perc_8.csv")
ref_apd_df = pd.read_csv("preprocessed_datasets/Regensburg_Pediatric_Appendicitis_unimputed.csv")
imputed_apd_df = pd.read_csv("preprocessed_datasets/Regensburg_Pediatric_Appendicitis_unimputed.csv")

In [45]:
feat_m1, feat_c1 = identify_binary_and_numerical_features(appedicities_MAR_15per_missing_dataset)

In [46]:
test_apd_MAR_mask_num, test_apd_MAR_mask_cat = generate_masks_for_missingness(
    ref_apd_df,
    test_apd_df_MAR,
    feat_m1,
    feat_c1,
    imputed_original_df = imputed_apd_df
)

In [47]:
test_apd_MNAR_mask_num, test_apd_MNAR_mask_cat = generate_masks_for_missingness(
    ref_apd_df,
    test_apd_df_MNAR,
    feat_m1,
    feat_c1,
    imputed_original_df = imputed_apd_df
)

##### Testing the correctness of these maps

In [48]:
test_apd_MAR_mask_num

array([[0, 2, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [2, 0, 0, ..., 0, 0, 2],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 2, 0, 2],
       [0, 0, 0, ..., 0, 0, 0]])

In [49]:
test_apd_df_MAR[list(feat_m1)]

,Neutrophil_Percentage,Height,Paedriatic_Appendicitis_Score,RBC_Count,Appendix_Diameter,BMI,WBC_Count,Hemoglobin,Length_of_Stay,CRP,Weight,Thrombocyte_Count,Age,Body_Temperature,RDW,Alvarado_Score
0,68.2,NaN,3.0,5.27,7.100000,16.90,7.7,14.8,3.0,0.0,37.0,254.0,12.68,37.0,12.2,4.0
1,64.8,147.0,4.0,5.26,7.596999,NaN,8.1,15.7,2.0,3.0,69.5,151.0,14.10,36.9,12.7,5.0
2,NaN,163.0,3.0,3.98,7.864903,23.30,NaN,11.4,4.0,3.0,62.0,300.0,14.14,36.6,12.2,NaN
3,63.0,165.0,6.0,4.64,7.195974,20.60,11.4,13.6,NaN,0.0,56.0,258.0,16.37,36.0,13.2,7.0
4,44.0,163.0,6.0,4.44,7.000000,16.90,8.1,12.6,3.0,0.0,45.0,311.0,11.08,36.9,13.6,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
708,58.5,151.0,6.0,4.49,6.926934,19.74,8.8,13.8,7.0,0.0,45.0,246.0,14.35,38.0,12.2,4.0
709,NaN,166.5,7.0,4.95,7.500000,25.25,11.4,NaN,NaN,71.0,70.0,NaN,12.41,39.4,NaN,8.0
710,68.5,152.0,3.0,4.49,7.387729,19.91,14.6,12.7,4.0,2.0,46.0,328.0,14.99,37.3,12.8,5.0
711,77.0,129.3,8.0,4.97,14.000000,14.30,NaN,14.3,5.0,8.0,23.9,345.0,7.20,NaN,12.7,NaN


In [50]:
test_apd_MAR_mask_cat

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 1, 1, 1],
       [0, 0, 0, ..., 1, 1, 1],
       ...,
       [0, 0, 2, ..., 1, 1, 1],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [51]:
test_apd_df_MAR[list(feat_c1)]

,Sex-is_female,Sex-is_male,Appendix_on_US-is_yes,Appendix_on_US-is_no,Migratory_Pain-is_no,Migratory_Pain-is_yes,Lower_Right_Abd_Pain-is_yes,Lower_Right_Abd_Pain-is_no,Contralateral_Rebound_Tenderness-is_yes,Contralateral_Rebound_Tenderness-is_no,...,Free_Fluids-is_no,Free_Fluids-is_yes,Appendix_Wall_Layers-is_intact,Appendix_Wall_Layers-is_raised,Appendix_Wall_Layers-is_upset,Appendix_Wall_Layers-is_partially raised,Surrounding_Tissue_Reaction-is_yes,Surrounding_Tissue_Reaction-is_no,Pathological_Lymph_Nodes-is_yes,Pathological_Lymph_Nodes-is_no
0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,NaN,NaN,NaN,NaN,1.000000,0.000000,1.000000,0.000000
1,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.615023,0.342723,0.004695,0.037559,0.821577,0.178423,0.765306,0.234694
2,1.0,0.0,0.0,1.0,NaN,NaN,NaN,NaN,1.0,0.0,...,1.0,0.0,NaN,NaN,NaN,NaN,0.821577,0.178423,0.765306,0.234694
3,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,...,1.0,0.0,0.615023,0.342723,0.004695,0.037559,NaN,NaN,1.000000,0.000000
4,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.615024,0.342723,0.004695,0.037558,0.821577,0.178423,1.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
708,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,...,1.0,0.0,0.615023,0.342723,0.004695,0.037559,0.821577,0.178423,0.765306,0.234694
709,NaN,NaN,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,...,1.0,0.0,0.000000,1.000000,0.000000,0.000000,NaN,NaN,0.765306,0.234694
710,1.0,0.0,NaN,NaN,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,1.0,0.615023,0.342723,0.004695,0.037559,0.821577,0.178423,0.765306,0.234694
711,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,...,0.0,1.0,NaN,NaN,NaN,NaN,1.000000,0.000000,0.000000,1.000000


In [52]:
test_apd_MNAR_mask_num

array([[0, 0, 2, ..., 2, 0, 2],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 2, 0],
       [0, 0, 0, ..., 2, 2, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [53]:
ref_apd_df[list(feat_m1)]

,Neutrophil_Percentage,Height,Paedriatic_Appendicitis_Score,RBC_Count,Appendix_Diameter,BMI,WBC_Count,Hemoglobin,Length_of_Stay,CRP,Weight,Thrombocyte_Count,Age,Body_Temperature,RDW,Alvarado_Score
0,68.2,148.0,3.0,5.27,7.1,16.90,7.7,14.8,3.0,0.0,37.0,254.0,12.68,37.0,12.2,4.0
1,64.8,147.0,4.0,5.26,NaN,31.90,8.1,15.7,2.0,3.0,69.5,151.0,14.10,36.9,12.7,5.0
2,74.8,163.0,3.0,3.98,NaN,23.30,13.2,11.4,4.0,3.0,62.0,300.0,14.14,36.6,12.2,5.0
3,63.0,165.0,6.0,4.64,NaN,20.60,11.4,13.6,3.0,0.0,56.0,258.0,16.37,36.0,13.2,7.0
4,44.0,163.0,6.0,4.44,7.0,16.90,8.1,12.6,3.0,0.0,45.0,311.0,11.08,36.9,13.6,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
708,58.5,151.0,6.0,4.49,NaN,19.74,8.8,13.8,7.0,0.0,45.0,246.0,14.35,38.0,12.2,4.0
709,76.6,166.5,7.0,4.95,7.5,25.25,11.4,13.7,4.0,71.0,70.0,243.0,12.41,39.4,13.4,8.0
710,68.5,152.0,3.0,4.49,NaN,19.91,14.6,12.7,4.0,2.0,46.0,328.0,14.99,37.3,12.8,5.0
711,77.0,129.3,8.0,4.97,14.0,14.30,17.8,14.3,5.0,8.0,23.9,345.0,7.20,37.5,12.7,9.0


In [54]:
test_apd_df_MNAR[list(feat_m1)]

,Neutrophil_Percentage,Height,Paedriatic_Appendicitis_Score,RBC_Count,Appendix_Diameter,BMI,WBC_Count,Hemoglobin,Length_of_Stay,CRP,Weight,Thrombocyte_Count,Age,Body_Temperature,RDW,Alvarado_Score
0,68.2,148.0,NaN,5.27,7.100000,16.90,7.7,NaN,3.0,NaN,37.0,254.0,12.68,NaN,12.2,NaN
1,64.8,147.0,4.0,5.26,NaN,NaN,NaN,15.7,NaN,3.0,NaN,151.0,NaN,36.9,12.7,5.0
2,74.8,163.0,3.0,3.98,7.864903,23.30,13.2,11.4,4.0,3.0,62.0,300.0,NaN,36.6,12.2,5.0
3,63.0,165.0,NaN,4.64,7.195974,20.60,NaN,NaN,3.0,NaN,56.0,258.0,16.37,36.0,13.2,NaN
4,44.0,163.0,6.0,4.44,7.000000,16.90,NaN,12.6,3.0,0.0,45.0,311.0,11.08,36.9,13.6,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
708,58.5,NaN,6.0,4.49,6.926934,19.74,NaN,13.8,NaN,NaN,45.0,246.0,14.35,NaN,12.2,4.0
709,76.6,NaN,7.0,4.95,7.500000,NaN,NaN,13.7,4.0,71.0,70.0,243.0,12.41,39.4,13.4,8.0
710,68.5,152.0,3.0,4.49,NaN,19.91,NaN,12.7,4.0,2.0,46.0,328.0,14.99,37.3,NaN,5.0
711,77.0,129.3,8.0,4.97,NaN,14.30,17.8,NaN,5.0,8.0,23.9,345.0,7.20,NaN,NaN,9.0


In [55]:
test_apd_MNAR_mask_cat

array([[0, 0, 0, ..., 2, 2, 2],
       [0, 0, 0, ..., 1, 1, 1],
       [0, 0, 0, ..., 1, 1, 1],
       ...,
       [0, 0, 0, ..., 1, 1, 1],
       [2, 2, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [56]:
test_apd_df_MNAR[list(feat_c1)]

,Sex-is_female,Sex-is_male,Appendix_on_US-is_yes,Appendix_on_US-is_no,Migratory_Pain-is_no,Migratory_Pain-is_yes,Lower_Right_Abd_Pain-is_yes,Lower_Right_Abd_Pain-is_no,Contralateral_Rebound_Tenderness-is_yes,Contralateral_Rebound_Tenderness-is_no,...,Free_Fluids-is_no,Free_Fluids-is_yes,Appendix_Wall_Layers-is_intact,Appendix_Wall_Layers-is_raised,Appendix_Wall_Layers-is_upset,Appendix_Wall_Layers-is_partially raised,Surrounding_Tissue_Reaction-is_yes,Surrounding_Tissue_Reaction-is_no,Pathological_Lymph_Nodes-is_yes,Pathological_Lymph_Nodes-is_no
0,1.0,0.0,1.0,0.0,NaN,NaN,1.0,0.0,1.0,0.0,...,NaN,NaN,1.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
1,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,NaN,NaN,...,1.0,0.0,0.615023,0.342723,0.004695,0.037559,0.821577,0.178423,0.765306,0.234694
2,1.0,0.0,0.0,1.0,1.0,0.0,NaN,NaN,1.0,0.0,...,1.0,0.0,0.615023,0.342723,0.004695,0.037559,0.821577,0.178423,0.765306,0.234694
3,1.0,0.0,0.0,1.0,NaN,NaN,1.0,0.0,0.0,1.0,...,1.0,0.0,0.615023,0.342723,0.004695,0.037559,0.821577,0.178423,1.000000,0.000000
4,1.0,0.0,NaN,NaN,NaN,NaN,1.0,0.0,1.0,0.0,...,NaN,NaN,0.615024,0.342723,0.004695,0.037558,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
708,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,NaN,NaN,...,NaN,NaN,0.615023,0.342723,0.004695,0.037559,0.821577,0.178423,0.765306,0.234694
709,1.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN,0.0,1.0,...,NaN,NaN,0.000000,1.000000,0.000000,0.000000,0.821577,0.178423,0.765306,0.234694
710,1.0,0.0,0.0,1.0,NaN,NaN,1.0,0.0,0.0,1.0,...,0.0,1.0,0.615023,0.342723,0.004695,0.037559,0.821577,0.178423,0.765306,0.234694
711,NaN,NaN,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,...,0.0,1.0,0.615024,0.342723,0.004695,0.037558,1.000000,0.000000,0.000000,1.000000


all of them are correct!

#### Testing <code>generate_masks_for_missingnes</code> using the <code>Heart Failure Clinical Records</code> (without pre-existing missingness)

##### Read the datasets

In [ ]:
%%bash
ls amputed_datasets/heart_failure_clinical_records

heart_failure_amputed_MAR_10_perc.csv
heart_failure_amputed_MAR_15_perc.csv
heart_failure_amputed_MAR_20_perc.csv
heart_failure_amputed_MAR_25_perc.csv
heart_failure_amputed_MAR_5_perc.csv
heart_failure_data_amputed_MNAR_10_perc.csv
heart_failure_data_amputed_MNAR_15_perc.csv
heart_failure_data_amputed_MNAR_20_perc.csv
heart_failure_data_amputed_MNAR_25_perc.csv
heart_failure_data_amputed_MNAR_5_perc.csv


In [ ]:
test_hf_df_MAR = pd.read_csv("amputed_datasets/heart_failure_clinical_records/heart_failure_amputed_MAR_20_perc.csv")
test_hf_df_MNAR = pd.read_csv("amputed_datasets/heart_failure_clinical_records/heart_failure_data_amputed_MNAR_15_perc.csv")
ref_hf_df = pd.read_csv("preprocessed_datasets/Heart_Failure_Clinical_Records_complete.csv")

In [ ]:
feat_m2, feat_c2 = identify_binary_and_numerical_features(test_hf_df_MNAR)

In [ ]:
test_hf_MAR_mask_num, test_hf_MAR_mask_cat = generate_masks_for_missingness(
    ref_hf_df,
    test_hf_df_MAR,
    feat_m2,
    feat_c2
)

No missingness!


In [ ]:
test_hf_MNAR_mask_num, test_hf_MNAR_mask_cat = generate_masks_for_missingness(
    ref_hf_df,
    test_hf_df_MNAR,
    feat_m2,
    feat_c2
)

No missingness!


##### Testing the correctness of these maps

In [ ]:
test_hf_df_MAR[list(feat_m2)]

,serum_sodium,time,creatinine_phosphokinase,serum_creatinine,platelets,age,ejection_fraction
0,130.0,4.0,582.0,1.9,265000.00,75.0,20.0
1,NaN,NaN,7861.0,NaN,263358.03,NaN,NaN
2,129.0,7.0,146.0,1.3,162000.00,65.0,20.0
3,137.0,7.0,111.0,1.9,210000.00,50.0,20.0
4,116.0,8.0,160.0,2.7,327000.00,65.0,NaN
...,...,...,...,...,...,...,...
294,143.0,270.0,61.0,1.1,155000.00,62.0,38.0
295,NaN,NaN,NaN,1.2,270000.00,NaN,38.0
296,NaN,NaN,NaN,NaN,742000.00,NaN,60.0
297,140.0,280.0,2413.0,1.4,140000.00,45.0,NaN


In [ ]:
test_hf_MAR_mask_num

array([[0, 0, 0, ..., 0, 0, 0],
       [2, 2, 0, ..., 0, 2, 2],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [2, 2, 2, ..., 0, 2, 0],
       [0, 0, 0, ..., 0, 0, 2],
       [0, 2, 0, ..., 0, 0, 0]])

In [ ]:
test_hf_df_MAR[list(feat_c2)]

,anaemia-is_no,anaemia-is_yes,diabetes-is_no,diabetes-is_yes,high_blood_pressure-is_no,high_blood_pressure-is_yes,sex-is_female,sex-is_male,smoking-is_no,smoking-is_yes
0,1.0,0.0,1.0,0.0,NaN,NaN,0.0,1.0,1.0,0.0
1,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0
2,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
3,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
4,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...
294,1.0,0.0,NaN,NaN,NaN,NaN,0.0,1.0,0.0,1.0
295,NaN,NaN,NaN,NaN,1.0,0.0,NaN,NaN,NaN,NaN
296,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
297,1.0,0.0,1.0,0.0,NaN,NaN,0.0,1.0,0.0,1.0


In [ ]:
test_hf_df_MAR[list(feat_c2)]

,anaemia-is_no,anaemia-is_yes,diabetes-is_no,diabetes-is_yes,high_blood_pressure-is_no,high_blood_pressure-is_yes,sex-is_female,sex-is_male,smoking-is_no,smoking-is_yes
0,1.0,0.0,1.0,0.0,NaN,NaN,0.0,1.0,1.0,0.0
1,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0
2,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
3,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
4,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...
294,1.0,0.0,NaN,NaN,NaN,NaN,0.0,1.0,0.0,1.0
295,NaN,NaN,NaN,NaN,1.0,0.0,NaN,NaN,NaN,NaN
296,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
297,1.0,0.0,1.0,0.0,NaN,NaN,0.0,1.0,0.0,1.0


In [ ]:
test_hf_df_MNAR[list(feat_m2)]

,serum_sodium,time,creatinine_phosphokinase,serum_creatinine,platelets,age,ejection_fraction
0,130.0,4.0,NaN,1.9,NaN,NaN,20.0
1,136.0,6.0,NaN,1.1,263358.03,55.0,38.0
2,129.0,7.0,146.0,NaN,162000.00,65.0,20.0
3,NaN,7.0,111.0,1.9,210000.00,50.0,20.0
4,116.0,8.0,160.0,2.7,327000.00,65.0,20.0
...,...,...,...,...,...,...,...
294,NaN,NaN,61.0,NaN,155000.00,62.0,38.0
295,139.0,NaN,1820.0,1.2,270000.00,55.0,38.0
296,NaN,NaN,2060.0,0.8,NaN,45.0,NaN
297,140.0,NaN,2413.0,1.4,140000.00,45.0,38.0


In [ ]:
test_hf_MNAR_mask_num

array([[0, 0, 2, ..., 2, 2, 0],
       [0, 0, 2, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [2, 2, 0, ..., 2, 0, 2],
       [0, 2, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [ ]:
test_hf_df_MNAR[list(feat_c2)]

,anaemia-is_no,anaemia-is_yes,diabetes-is_no,diabetes-is_yes,high_blood_pressure-is_no,high_blood_pressure-is_yes,sex-is_female,sex-is_male,smoking-is_no,smoking-is_yes
0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,NaN,NaN
1,NaN,NaN,1.0,0.0,1.0,0.0,0.0,1.0,NaN,NaN
2,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
3,NaN,NaN,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
4,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
294,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0
295,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
296,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0
297,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0


In [ ]:
test_hf_MNAR_mask_cat

array([[0, 0, 0, ..., 0, 2, 2],
       [2, 2, 0, ..., 0, 2, 2],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [ ]:
np.any(test_hf_MAR_mask_num==1)

False

In [ ]:
np.any(test_hf_MAR_mask_cat==1)

False

In [ ]:
np.any(test_hf_MNAR_mask_num==1)

False

In [ ]:
np.any(test_hf_df_MNAR_cat==1)

False

The masking function applies perfectly to the as datasets without missingness well!

## Normalize and Pre-impute the five reference datasets using BRR, and the 500 amputed datasets

In [57]:
%%bash
ls .

amputed_datasets
datasets_masking_and_normalization.ipynb
individual_datasets_preprocessing.ipynb
preprocessed_datasets
preprocessing_Awan_2022_datasets


Test saving a random image to google drive

Write the method to obtain the normalized and RF or BRR-imputed datasets (500 * 3) in total

In [70]:
# 1) set up the imputer to use BayesianRidge
default_BRR_imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=10,          # number of imputation rounds, the default value = 300, switched to 10 here for faster training
    tol=1e-3,             # convergence tolerance
    random_state=42,
)

In [72]:
# 2) set up the imputer to use BayesianRidge
# inside each process we build a fresh imputer
default_RF_imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_jobs=-1),
    max_iter=10,
    tol=1e-3,
    random_state=42,
)

## Batch-normalize all the datasets

In [83]:
%%bash 
ls ./amputed_datasets/appendicitis_data/MAR_15_perc_4.csv

./amputed_datasets/appendicitis_data/MAR_15_perc_4.csv


In [119]:
BASE_DIR = "./amputed_datasets"

In [120]:
os.path.exists(BASE_DIR)

True

In [105]:
%%bash
mkdir -p ../../VAE_Q_learning_imputation_baseline/imputed_datasets

In [106]:
'''
An example of the input args:
    input_root == "/work/jiz_imputation/VAEQL_Imputation/VAE_Q_learning_imputation_baseline/datasets_preprocessing/amputed_datasets"
    input_ds_name == "appendicitis_data"
    input_ds_subname == "MAR_15_perc_4.csv"
    output_root_folder == "/work/jiz_imputation/VAEQL_Imputation/VAE_Q_learning_imputation_baseline/imputed_datasets"
    imputer == IterativeImputer(
        estimator=RandomForestRegressor(),
        max_iter=10,          
        tol=1e-3,
        random_state=42
    ),
    imputation_method_name == "RF"
'''

def impute_the_amputed_dataset_and_export(
    *,
    input_root: str, 
    input_ds_name: str,
    input_ds_subname: str,
    output_root_folder: str,
    imputer: IterativeImputer,
    imputation_method_name: str
) -> str:

    if "." not in input_ds_subname or len(input_ds_subname.split(".")) != 2:
        raise ValueError("input_ds_subname must be like 'xxx.csv'.")

    amputed_path = f"{input_root}/{input_ds_name}/{input_ds_subname}"
    amputed_df = pd.read_csv(amputed_path)
    num_feats, _ = identify_binary_and_numerical_features(amputed_df)

    output_dir = os.path.join(output_root_folder, input_ds_name)
    os.makedirs(output_dir, exist_ok=True)

    output_prefix = input_ds_subname.split(".")[0]
    norm_amputed_df_path = os.path.join(output_dir, f"{output_prefix}_NORM.csv")
    temp_path = norm_amputed_df_path + ".tmp"
    lock_path = norm_amputed_df_path + ".lock"

    output_path = os.path.join(output_dir, f"{output_prefix}_{imputation_method_name}.csv")
    if os.path.exists(output_path):
        pass

    # --- lock normalization step ---
    with FileLock(lock_path, timeout=180):  # wait up to 3 minutes if another proc holds it
        if os.path.exists(norm_amputed_df_path):
            try:
                normalized_amputed_df = pd.read_csv(norm_amputed_df_path)
                if normalized_amputed_df.empty:
                    raise pd.errors.EmptyDataError
            except pd.errors.EmptyDataError:
                print(f"[Warning] Empty normalization file detected: {norm_amputed_df_path}, rebuilding...")
                normalized_amputed_df = normalize_numerical_features(amputed_df, num_feats)
                normalized_amputed_df.to_csv(temp_path, index=False)
                os.replace(temp_path, norm_amputed_df_path)
        else:
            normalized_amputed_df = normalize_numerical_features(amputed_df, num_feats)
            normalized_amputed_df.to_csv(temp_path, index=False)
            os.replace(temp_path, norm_amputed_df_path)

    # --- imputation ---
    imputed_array = imputer.fit_transform(normalized_amputed_df)
    imputed_df = pd.DataFrame(imputed_array, columns=amputed_df.columns, index=amputed_df.index)

    imputed_df.to_csv(output_path, index=False)

    return output_path

In [107]:
csv_paths = []
for root, _, files in os.walk(base_dir):
    for f in files:
        if f.endswith(".csv"):
            csv_paths.append(os.path.join(root, f))

print(f"Found {len(csv_paths)} CSV files in all subfolders.")

Found 500 CSV files in all subfolders.


In [108]:
csv_paths[0].split("/")

['.', 'amputed_datasets', 'HCV_Egyptian_patients', 'MAR_5_perc_3.csv']

In [109]:
# worker function for parallel jobs
def imputation_worker(kwargs):
    output_path = impute_the_amputed_dataset_and_export(**kwargs)
    return f"✅ Done: {kwargs['input_ds_name']}/{os.path.basename(output_path)}"

## Perform the parallelized RF and BRR imputation

In [110]:
imputers_dict: dict[str, pd.DataFrame] = {
    "RF": default_RF_imputer,
    "BRR": default_BRR_imputer,
}

In [112]:
OUTPUT_ROOT = "../../VAE_Q_learning_imputation_baseline/imputed_datasets"
os.path.exists(OUTPUT_ROOT)

True

In [ ]:
job_kwargs_list: List[Dict[str, Any]] = []

for csv_path in csv_paths:

    csv_path_comps = csv_path.split("/")
    
    for k, v in imputers_dict.items():
        args_dict = dict(
            input_root=BASE_DIR,
            input_ds_name = csv_path_comps[-2],
            input_ds_subname = csv_path_comps[-1],
            output_root_folder = OUTPUT_ROOT,
            imputer=v,
            imputation_method_name=k
        )
        job_kwargs_list.append(args_dict)


# --- run multiprocessing ---
num_cores = min(60, cpu_count()-2)  # use up to 60 or however many are available minus 2
with Pool(processes=num_cores) as pool:
    for result in pool.imap_unordered(imputation_worker, job_kwargs_list):
        print(result)
